# Lab 02 — ReAct Agent Loop from Scratch

**Pairs with:** [Course 07 · AI Agents](https://psssnikhil.github.io/ai-engineering-handbook/build/module-11-ai-agents-fundamentals/) and [Agent Engineering Track](https://psssnikhil.github.io/ai-engineering-handbook/agent-engineering/)

In this lab, you will build an autonomous **ReAct (Reasoning + Acting) Agent Loop** from scratch in pure Python with **no frameworks**.

You will implement:
1. **Tool Registry** — Registering Python functions and generating tool definitions.
2. **The ReAct Loop** — Model generation → action parsing → tool execution → observation feedback.
3. **Error Recovery** — Gracefully handling invalid tool arguments or unknown action calls.
4. **Termination Guards** — Enforcing max iterations and detecting `Final Answer:` tags.

```
User Query ──► ReAct Loop: [Thought ──► Action (Tool) ──► Observation] ──► Final Answer
```

**Prerequisites:** `pip install -r requirements.txt` and `ANTHROPIC_API_KEY` set in your environment.

In [ ]:
import os
import json
import math
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-3-5-sonnet-20241022"

## Step 1: Define Tools & Tool Registry

An agent needs tools to interact with the environment. Let's create two mock tools: a calculator and a database lookup tool.

In [ ]:
# Mock tools
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression."""
    try:
        # Safe math evaluation
        allowed = {"math": math, "abs": abs, "round": round}
        result = eval(expression, {"__builtins__": None}, allowed)
        return str(result)
    except Exception as e:
        return f"Calculation Error: {e}"

def lookup_user_account(user_id: str) -> str:
    """Looks up user account status and balance."""
    database = {
        "usr_101": {"name": "Alice", "tier": "Premium", "balance": 450.00},
        "usr_102": {"name": "Bob", "tier": "Free", "balance": 12.50}
    }
    user = database.get(user_id)
    if user:
        return json.dumps(user)
    return f"Error: User ID '{user_id}' not found in database."

# Tool Dispatcher Registry
TOOL_REGISTRY = {
    "calculate": calculate,
    "lookup_user_account": lookup_user_account
}

## Step 2: System Prompt & ReAct Protocol

We instruct the model to follow the ReAct prompt template: outputting `Thought:`, `Action:`, and waiting for `Observation:`.

In [ ]:
SYSTEM_PROMPT = """You are a ReAct AI Agent. You solve user queries step by step using tools.

Available Tools:
- calculate(expression): Evaluates math expression string, e.g. calculate(expression="450.0 * 0.15")
- lookup_user_account(user_id): Looks up user details, e.g. lookup_user_account(user_id="usr_101")

Use the following exact format:

Thought: Describe your reasoning process.
Action: {"name": "tool_name", "args": {"arg_name": "value"}}
Observation: <tool output will be provided here>
... (repeat Thought/Action/Observation if needed)
Thought: I know the final answer.
Final Answer: Your response to the user.
"""

## Step 3: Implement the ReAct Loop Execution

Now we write the execution loop that handles LLM outputs, dispatches tool execution, and appends `Observation:` back into the message history.

In [ ]:
def run_react_agent(user_query: str, max_turns: int = 5):
    messages = [{"role": "user", "content": user_query}]
    print(f"=== Starting ReAct Agent for Query: '{user_query}' ===\n")
    
    for turn in range(1, max_turns + 1):
        print(f"--- Turn {turn} ---")
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            system=SYSTEM_PROMPT,
            messages=messages
        )
        
        llm_text = response.content[0].text
        print(llm_text)
        messages.append({"role": "assistant", "content": llm_text})
        
        # 1. Check for Final Answer
        if "Final Answer:" in llm_text:
            final_ans = llm_text.split("Final Answer:")[1].strip()
            print(f"\n✓ Agent Completed in {turn} turns.")
            return final_ans
            
        # 2. Check for Action call
        if "Action:" in llm_text:
            try:
                action_line = llm_text.split("Action:")[1].strip().split("\n")[0]
                action_data = json.loads(action_line)
                tool_name = action_data["name"]
                tool_args = action_data.get("args", {})
                
                if tool_name in TOOL_REGISTRY:
                    obs = TOOL_REGISTRY[tool_name](**tool_args)
                else:
                    obs = f"Error: Tool '{tool_name}' does not exist."
            except Exception as e:
                obs = f"Error parsing action JSON: {e}"
                
            obs_msg = f"Observation: {obs}"
            print(f"-> Executed Tool | {obs_msg}\n")
            messages.append({"role": "user", "content": obs_msg})
            
    return "Error: Max turns reached without final answer."

## Step 4: Run Agent Example

Let's test the agent on a query requiring two steps: database lookup + math calculation.

In [ ]:
query = "Look up account usr_101. Calculate what their balance would be after a 15% bonus."
result = run_react_agent(query)
print(f"\nFINAL RESULT:\n{result}")